In [ ]:
import os
import torch
import torchvision
import transformers

# 继承CocoDetection，实现一些定制处理的数据集
class CocoDetection(torchvision.datasets.CocoDetection):
    def __init__(self, ds_folder, processor, train=True):
        # 标注文件
        ann_file = os.path.join(ds_folder, "train.json" if train else "val.json")
        # 图像目录
        img_dir = os.path.join(ds_folder, "train" if train else "val")
        super(CocoDetection, self).__init__(img_dir, ann_file)
        self.processor = processor

    def __getitem__(self, idx):
        # 读取PIL image和标签target（默认格式），我们需要进行转换（比如转换为张量）
        img, target = super(CocoDetection, self).__getitem__(idx)
        
        # 转换图像与标签格式 (重点是把标签转换为训练模式支持的格式DETR, 同时缩放，标准化图像与标签
        # 把我们需要访问的图像索引，映射到对应的图像编号，保证能正确访问到文件
        image_id = self.ids[idx]  
        target = {'image_id': image_id, 'annotations': target}
         # 全部转换为张量，而且只获取训练需要的数据。
        encoding = self.processor(images=img, annotations=target, return_tensors="pt")
        pixel_values = encoding["pixel_values"].squeeze() # 返回的数据按照torch的格式有批次维度，去掉批次维度
        target = encoding["labels"][0] # 使用下标的方式去掉维度
        return pixel_values, target
        
# 创建批次数据集
# processor = transformers.YolosImageProcessor.from_pretrained("F:/03Models/yolos-tiny", size=512, max_size=1333)
processor = transformers.YolosImageProcessor.from_pretrained("E:/Models/yolos-tiny")
train_dataset = CocoDetection(ds_folder='E:/Datasets/balloon/', processor=processor)
val_dataset = CocoDetection(ds_folder='E:/Datasets/balloon/', processor=processor, train=False)

def collate_fn(batch):
    # 去除图像，进行对齐
    pixel_values = [item[0] for item in batch]
    encoding = processor.pad(pixel_values, return_tensors="pt")
    # 取出标签
    labels = [item[1] for item in batch]
    # 重新返回处理过的数据集
    batch = {}
    batch['pixel_values'] = encoding['pixel_values']
    batch['labels'] = labels
    return batch

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# 类别数量改变，其分类器也会被强制改变，原来的矩阵权重就加载不进去，这个部分的权重会被从头训练。
model = transformers.YolosForObjectDetection.from_pretrained(
    "F:/03Models/yolos-tiny", 
    num_labels=len(train_dataset.coco.cats),  # 类别数量（我们这类是1）
    ignore_mismatched_sizes=True
)  # 注意：加载的模型可能在cpu上（这个模型就是在cpu上）
model.to(device)

epochs = 1000
learning_rate = 2.5e-5   # 学习率
weight_decay=1e-4   # 权重惩罚系数（也称权重衰减系数）
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=learning_rate,
    weight_decay=weight_decay
)